# 🇧🇩 Bangla Distill — Unsloth Fast Inference Runner on Kaggle

Runs the distilled 500M student model (`Qwen2.5-0.5B-Instruct` + Bengali LoRA Adapter) using **Unsloth's native inference acceleration** (up to 2x faster token generation).

> **GPU:** Kaggle P100 or T4 GPU (Runtime: GPU enabled).

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install -q --no-deps "xformers<0.0.29" "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes sentencepiece

## Step 2: Download or Locate LoRA Adapter

In [ ]:
import os
import urllib.request

ADAPTER_DIR = "/kaggle/working/adapter" if os.path.exists("/kaggle") else "./adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Search /kaggle/input first
found_locally = False
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_model.safetensors" in files:
            ADAPTER_DIR = root
            found_locally = True
            print(f"✅ Found adapter in Kaggle Input: {ADAPTER_DIR}")
            break

# If not uploaded, download directly from GitHub repo
if not found_locally and not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_model.safetensors")):
    print("Downloading adapter files directly from GitHub...")
    BASE_URL = "https://raw.githubusercontent.com/AshmitNur/Distilation/main/final-lora-adapter-v2"
    files_to_download = [
        "adapter_config.json",
        "chat_template.jinja",
        "tokenizer.json",
        "tokenizer_config.json",
        "TRAINING_INFO.txt"
    ]
    for fn in files_to_download:
        try:
            urllib.request.urlretrieve(f"{BASE_URL}/{fn}", os.path.join(ADAPTER_DIR, fn))
        except Exception as e:
            pass
            
    # Download safetensors weights (via GitHub media)
    weight_url = "https://github.com/AshmitNur/Distilation/raw/main/final-lora-adapter-v2/adapter_model.safetensors"
    print("Downloading adapter_model.safetensors (~35 MB)...")
    urllib.request.urlretrieve(weight_url, os.path.join(ADAPTER_DIR, "adapter_model.safetensors"))
    print("✅ Download complete!")

print(f"\nAdapter path: {ADAPTER_DIR}")

## Step 3: Load Model with Unsloth Fast Inference

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load base model & attach adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR, # Unsloth automatically loads base + LoRA adapter!
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)

# Enable Unsloth 2x fast inference mode
FastLanguageModel.for_inference(model)
print("✅ Model loaded and ready for fast inference!")

## Step 4: Interactive Bengali Chat Function

In [ ]:
def ask_bangla(prompt, temperature=0.3, top_p=0.9, max_tokens=256):
    messages = [
        {"role": "system", "content": "আপনি একজন বাংলা ভাষায় দক্ষ সহকারী। ব্যবহারকারীর প্রশ্নের উত্তর সঠিক তথ্যসহ বিস্তারিতভাবে বাংলায় দিন।"},
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    return response

# Test query
query = "বাংলাদেশের জাতীয় কবি কাজী নজরুল ইসলাম সম্পর্কে বিস্তারিত বলুন।"
print(f"প্রশ্ন: {query}\n")
print(f"উত্তর:\n{ask_bangla(query)}")